In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lecture 9 - Hyperparameter Tuning in Random Forest

## Outline

1. **Introduction**
    - What is Hyperparameter Tuning?
    - Importance of Hyperparameter Tuning in Machine Learning
    - Focus on Random Forest Algorithm
    - Dataset Overview: Diabetes Dataset from scikit-learn

2. **Preparing the Data**
    - Loading the Diabetes Dataset
    - Data Preprocessing

3. **Basic Random Forest Model**
    - Building a Basic Random Forest Model
    - Model Evaluation

4. **GridSearch Technique**
    - Introduction to GridSearch
    - Implementing GridSearch with Random Forest
    - Evaluating the Best Model

5. **RandomSearch Technique**
    - Introduction to RandomSearch
    - Implementing RandomSearch with Random Forest
    - Evaluating the Best Model

6. **Nested Cross-Validation**
    - What is Nested Cross-Validation?
    - Implementing Nested CV with GridSearch and Random Forest
    - Implementing Nested CV with RandomSearch and Random Forest

7. **Comparing Techniques**
    - GridSearch vs RandomSearch
    - With and Without Nested CV

8. **Best Practices**
    - Tips for Effective Hyperparameter Tuning

9. **Exercises**
    - Detailed exercises for hands-on practice

10. **Solutions to Exercises**
    - Detailed solutions to each exercise

## Introduction

### What is Hyperparameter Tuning?

Hyperparameter tuning is the process of systematically searching for the best combination of hyperparameters that optimize a machine learning model's performance for a given problem. Unlike model parameters, which are learned during training, hyperparameters are set by the data scientist before training begins.

### Importance of Hyperparameter Tuning in Machine Learning

Hyperparameter tuning is crucial for achieving optimal model performance. Incorrectly set hyperparameters can lead to models that are either overfitting or underfitting. Therefore, it's essential to find the right set of hyperparameters to train a robust and accurate model.

### Focus on Random Forest Algorithm

Random Forest is an ensemble learning method that can be used for both classification and regression tasks. It is flexible and capable of achieving excellent performance with minimal hyperparameter tuning. However, fine-tuning its hyperparameters can lead to even better results.

### Dataset Overview: Diabetes Dataset from scikit-learn

The diabetes dataset is used for regression. It contains ten baseline variables (age, sex, BMI, average blood pressure, and six blood serum measurements) for 442 diabetes patients. The target is a quantitative measure of disease progression one year after baseline.

### Feature Descriptions




In [11]:
import warnings
warnings.filterwarnings('ignore')
# Importing necessary libraries
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import numpy as np

# Loading the diabetes dataset
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target

# Splitting the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [2]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641
...,...,...,...,...,...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,-0.005697,-0.002566,-0.028674,-0.002592,0.031193,0.007207
438,-0.005515,0.050680,-0.015906,-0.067642,0.049341,0.079165,-0.028674,0.034309,-0.018114,0.044485
439,0.041708,0.050680,-0.015906,0.017293,-0.037344,-0.013840,-0.024993,-0.011080,-0.046883,0.015491
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044529,-0.025930


In [3]:
pd.DataFrame(y)

,0
0,151.0
1,75.0
2,141.0
3,206.0
4,135.0
...,...
437,178.0
438,104.0
439,132.0
440,220.0


## Basic Random Forest Model

In this section, we will build a basic Random Forest model without any hyperparameter tuning. This will serve as our baseline model for comparison.

### Building a Random Forest Model

Random Forest algorithm is an ensemble learning technique that amalgamates multiple decision trees to construct a more robust and generalizable model. The algorithm's efficacy is contingent on several hyperparameters, each of which plays a pivotal role in shaping the model's performance. Below are some of the key features and their implications:

1. **Number of Trees (`n_estimators`)**: The 'forest' in Random Forest is essentially a collection of decision trees. While increasing the number of trees generally enhances the model's performance by reducing overfitting, there is a point of diminishing returns. Beyond this point, adding more trees neither significantly improves the model's predictive power nor its generalizability but does augment computational complexity.

2. **Maximum Depth of Trees (`max_depth`)**: This parameter controls the depth of each tree in the forest. A shallower tree is computationally less expensive but may underfit the data, while a deeper tree may capture more complex patterns but risks overfitting. Optimal tuning of this parameter is crucial for balancing bias and variance.

3. **Maximum Features (`max_features`)**: This parameter specifies the maximum number of features each tree is allowed to use when splitting a node. A lower value reduces the risk of overfitting but may miss out on capturing important patterns. Conversely, a higher value may capture more complex relationships but increases the risk of overfitting.

4. **Minimum Samples per Leaf (`min_samples_leaf`)**: This parameter sets the minimum number of samples required to be at a leaf node. Smaller leaf sizes make the model more prone to capturing noise in the data, thus overfitting. Conversely, larger leaf sizes, while more robust, may fail to capture essential data patterns.

5. **Bootstrap Sampling (`bootstrap`)**: This is a boolean parameter that decides whether or not bootstrap samples are used when building trees. If set to `False`, the whole dataset is used to build each tree, making the ensemble of trees more correlated and potentially increasing the variance of the model.

6. **Criterion (`criterion`)**: This parameter defines the function to measure the quality of a split. Common choices are "gini" for the Gini impurity and "entropy" for the information gain. The choice of criterion can subtly influence the partitions and, consequently, the model's performance.

7. **Node Splitting (`min_samples_split`)**: This parameter specifies the minimum number of samples required to split an internal node. Like `min_samples_leaf`, it can help control the complexity of the trees and, thus, the risk of overfitting.


### Model Evaluation

We will use the Root Mean Square Error (RMSE) as the evaluation metric. RMSE gives us an idea of how much error the system typically makes in its predictions, with a higher weight for large errors.

In [4]:
# Importing additional libraries for model evaluation
from sklearn.metrics import mean_squared_error
from math import sqrt

# Building a basic Random Forest model
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

# Making predictions
y_pred = rf.predict(X_test)

# Evaluating the model
rmse = sqrt(mean_squared_error(y_test, y_pred))
print(f'Root Mean Square Error: {rmse}')

Root Mean Square Error: 54.332408273184846


## GridSearch Technique

### Introduction to GridSearch

GridSearch is a hyperparameter tuning technique that exhaustively searches through a manually specified subset of the hyperparameter space. It performs k-fold cross-validation for each hyperparameter combination and returns the best combination based on the cross-validation performance.

### Implementing GridSearch with Random Forest

We will use scikit-learn's `GridSearchCV` to perform GridSearch. The hyperparameters we will tune are `n_estimators`, `max_features`, and `max_depth`.

### Evaluating the Best Model

After finding the best hyperparameters, we will evaluate the model's performance using RMSE.

In [12]:
# Importing GridSearchCV
from sklearn.model_selection import GridSearchCV

# Defining the hyperparameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth': [10, 20, 30, None]
}

# Creating the GridSearchCV object
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

# Fitting the model
grid_search.fit(X_train, y_train)

# Getting the best parameters
best_params = grid_search.best_params_
print(f'Best Parameters: {best_params}')

# Evaluating the best model
best_grid = grid_search.best_estimator_
grid_pred = best_grid.predict(X_test)
grid_rmse = sqrt(mean_squared_error(y_test, grid_pred))
print(f'Root Mean Square Error of Best Model: {grid_rmse}')

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best Parameters: {'max_depth': 30, 'max_features': 'sqrt', 'n_estimators': 200}
Root Mean Square Error of Best Model: 53.73969824194058


**Exercise**: Implement GridSearch with Random Forest using a different set of hyperparameter values. Compare the RMSE with the previous GridSearch model.


**Exercise**: Implement GridSearch with Random Forest using a different set of hyperparameters and values. Compare the with previous results.


## RandomSearch Technique

### Introduction to RandomSearch

Unlike GridSearch, which is exhaustive, RandomSearch samples a fixed number of hyperparameter combinations from a distribution for each hyperparameter. This makes RandomSearch computationally less expensive and often finds a good set of hyperparameters faster.

### Implementing RandomSearch with Random Forest

We will use scikit-learn's `RandomizedSearchCV` for this. The hyperparameters and their distributions are similar to those in GridSearch.

### Evaluating the Best Model

After finding the best hyperparameters using RandomSearch, we will evaluate the model's performance using RMSE.

In [13]:
# Importing RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Defining the hyperparameter distribution
param_dist = {
    'n_estimators': randint(50, 200),
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth': randint(1, 40)
}

# Creating the RandomizedSearchCV object
random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_dist, n_iter=100, cv=5, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error', random_state=42)

# Fitting the model
random_search.fit(X_train, y_train)

# Getting the best parameters
best_random_params = random_search.best_params_
print(f'Best Parameters: {best_random_params}')

# Evaluating the best model
best_random = random_search.best_estimator_
random_pred = best_random.predict(X_test)
random_rmse = sqrt(mean_squared_error(y_test, random_pred))
print(f'Root Mean Square Error of Best Model: {random_rmse}')

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best Parameters: {'max_depth': 5, 'max_features': 'sqrt', 'n_estimators': 191}
Root Mean Square Error of Best Model: 53.09294665882955


**Exercise**: Implement RandomSearch with Random Forest using a different set of hyperparameters. Compare the RMSE with the previous RandomSearch and GridSearch models.


## Nested Cross-Validation

### What is Nested Cross-Validation?

Nested Cross-Validation (Nested CV) is a technique used to prevent the model evaluation from being overly optimistic. It involves an 'inner' CV loop for hyperparameter tuning and an 'outer' CV loop for model evaluation.

### Implementing Nested CV with GridSearch and Random Forest

We will implement Nested CV using scikit-learn's `cross_val_score` with `GridSearchCV` as the estimator.

### Implementing Nested CV with RandomSearch and Random Forest

Similarly, we will implement Nested CV using `cross_val_score` with `RandomizedSearchCV` as the estimator.

Breaking it down:

1. **Inner Loop**: The `GridSearchCV` and `RandomizedSearchCV` objects (`grid_search` and `random_search`) are essentially running an inner cross-validation loop. When you fit these objects to the data, they perform hyperparameter tuning by splitting the data into training and validation sets multiple times (defined by their `cv` parameter).

2. **Outer Loop**: The `cross_val_score` function performs an outer cross-validation loop. It takes the entire dataset (`X, y`), splits it into training and test sets multiple times (defined by its `cv` parameter), and then fits and evaluates the model on these splits.

3. **Nested Nature**: For each split in the outer loop, the inner loop performs a full hyperparameter tuning process. This ensures that the hyperparameters are tuned based on a portion of the data that is entirely separate from the portion used for testing, thereby providing a more unbiased estimate of the model's performance on unseen data.

So, in essence, for each fold in the outer CV loop, the inner CV loop finds the best hyperparameters using only the training data for that fold. Then, the performance is evaluated on the test data for that fold. This process is repeated for each fold in the outer loop, and the results are averaged to get a more robust performance metric. e nested structure is implicit in the combination of `cross_val_score` with `GridSearchCV` or `RandomizedSearchCV`. The training set in each fold of the outer loop is indeed split further by the inner loop, fulfilling the criteria for nested cross-validation.

In [14]:
# Importing cross_val_score for Nested CV
from sklearn.model_selection import cross_val_score

# Implementing Nested CV with GridSearch and Random Forest
nested_grid_score = cross_val_score(grid_search, X, y, cv=5, scoring='neg_mean_squared_error')
nested_grid_rmse = np.sqrt(-nested_grid_score.mean())
print(f'Nested CV RMSE with GridSearch: {nested_grid_rmse}')

# Implementing Nested CV with RandomSearch and Random Forest
nested_random_score = cross_val_score(random_search, X, y, cv=5, scoring='neg_mean_squared_error')
nested_random_rmse = np.sqrt(-nested_random_score.mean())
print(f'Nested CV RMSE with RandomSearch: {nested_random_rmse}')

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Nested CV RMSE with GridSearch: 56.70083898126356
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Nested CV RMSE with RandomSearch: 56.80748400518606


## Comparing Techniques

### GridSearch vs RandomSearch

Both GridSearch and RandomSearch have their merits and demerits. GridSearch is exhaustive but computationally expensive. RandomSearch is less accurate but faster. The choice between the two often depends on the specific needs of a project.

### With and Without Nested CV

Nested CV provides a more unbiased evaluation but is computationally more intensive. It is advisable to use Nested CV when the final model evaluation's reliability is crucial.

**Exercise**: Use Nested CV with GridSearch but change the number of folds in the outer loop. Observe the change in RMSE.



In [15]:
# Creating the GridSearchCV object -internal loop
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

# Implementing Nested CV with GridSearch and Random Forest  external loop
nested_grid_score = cross_val_score(grid_search, X, y, cv=3, scoring='neg_mean_squared_error')
nested_grid_rmse = np.sqrt(-nested_grid_score.mean())
print(f'Nested CV RMSE with GridSearch: {nested_grid_rmse}')

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Nested CV RMSE with GridSearch: 56.92830251747671


**Exercise**: Use Nested CV with RandomSearch but change the number of folds in the outer loop. Observe the change in RMSE.

In [16]:
# Creating the GridSearchCV object
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

# Implementing Nested CV with GridSearch and Random Forest
nested_grid_score = cross_val_score(grid_search, X, y, cv=5, scoring='neg_mean_squared_error')
nested_grid_rmse = np.sqrt(-nested_grid_score.mean())
print(f'Nested CV RMSE with GridSearch: {nested_grid_rmse}')

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Nested CV RMSE with GridSearch: 56.90036513539102


## Best Practices

1. **Start Simple**: Always start with a simple model and default hyperparameters. This serves as a baseline for comparison.

2. **Understand the Hyperparameters**: Before tuning, understand what each hyperparameter does. This will help in setting the right range or distribution for each hyperparameter.

3. **Use Parallel Computing**: Both GridSearch and RandomSearch allow parallel computing (`n_jobs=-1`). This speeds up the tuning process.

4. **Random State**: For reproducibility, set a random state where applicable.

5. **Cross-Validation**: Always use cross-validation for a more robust evaluation.

6. **Nested CV for Final Evaluation**: If computational resources allow, use Nested CV for the final model evaluation.

## Exercises


**Exercise**: Implement GridSearch with a different machine learning algorithm (e.g., Support Vector Machine). Compare the RMSE with the Random Forest model.

**Exercise**: Use a different evaluation metric (e.g., MAE) for GridSearch and compare it with RMSE.

**Exercise**: Implement GridSearch without cross-validation (cv=1). Observe the change in RMSE and explain why it changed.


In [22]:
import xgboost as xgb

xg = xgb.XGBRegressor(random_state=42)
xg.fit(X_train, y_train)

# Making predictions
y_pred = xg.predict(X_test)

# Evaluating the model
rmse = sqrt(mean_squared_error(y_test, y_pred))
print(f'Root Mean Square Error: {rmse}')


# Defining the hyperparameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    # 'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth': [10, 20, 30, None]
}

grid_search = GridSearchCV(estimator=xg, param_grid=param_grid, cv = 2, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

# Fitting the model
grid_search.fit(X_train, y_train)

# Getting the best parameters
best_params = grid_search.best_params_
print(f'Best Parameters: {best_params}')

# Evaluating the best model
best_grid = grid_search.best_estimator_
grid_pred = best_grid.predict(X_test)
grid_rmse = sqrt(mean_squared_error(y_test, grid_pred))
print(f'Root Mean Square Error of Best Model: {grid_rmse}')

Root Mean Square Error: 57.88783670048563
Fitting 2 folds for each of 12 candidates, totalling 24 fits
Best Parameters: {'max_depth': None, 'n_estimators': 50}
Root Mean Square Error of Best Model: 57.96540466050509


In [21]:
from sklearn.metrics import mean_absolute_error

grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv = 5, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

# Fitting the model
grid_search.fit(X_train, y_train)

# Getting the best parameters
best_params = grid_search.best_params_
print(f'Best Parameters: {best_params}')

# Evaluating the best model
best_grid = grid_search.best_estimator_
grid_pred = best_grid.predict(X_test)
grid_mae = mean_absolute_error(y_test, grid_pred)
print(f'Mean Absolute Error of Best Model: {grid_mae}')

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best Parameters: {'max_depth': 30, 'max_features': 'sqrt', 'n_estimators': 200}
Mean Absolute Error of Best Model: 43.53544943820225


In [20]:
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv = 2, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

# Fitting the model
grid_search.fit(X_train, y_train)

# Getting the best parameters
best_params = grid_search.best_params_
print(f'Best Parameters: {best_params}')

# Evaluating the best model
best_grid = grid_search.best_estimator_
grid_pred = best_grid.predict(X_test)
grid_rmse = sqrt(mean_squared_error(y_test, grid_pred))
print(f'Root Mean Square Error of Best Model: {grid_rmse}')

Fitting 2 folds for each of 36 candidates, totalling 72 fits
Best Parameters: {'max_depth': 20, 'max_features': 'sqrt', 'n_estimators': 50}
Root Mean Square Error of Best Model: 53.506646610123816
